# Inspect Ingested Data Tables

This notebook helps you inspect the SQLite bronze tables used by Skimmer.

- Default DB path: `data/skimmer.db`
- Override with env var: `SKIMMER_DB_PATH`
- It lists tables, shows schema, row counts, and recent records.

In [9]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    """Locate the repository root so the notebook runs from any working directory."""

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'RecomendationAnalysis').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError('Could not locate the skimmer repository root from the current directory.')


# The default database path is repository-relative, so it must not depend on the
# kernel's working directory: this notebook lives in notebooks/, where a bare
# 'data/skimmer.db' resolves to notebooks/data/skimmer.db and does not exist.
REPO_ROOT = find_repo_root()
DEFAULT_DB_PATH = REPO_ROOT / 'data' / 'skimmer.db'
DB_PATH = Path(os.environ.get('SKIMMER_DB_PATH', DEFAULT_DB_PATH)).expanduser()

print(f'Repository root: {REPO_ROOT}')
print(f'Using database: {DB_PATH.resolve()}')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')

Repository root: /home/orangepi/code_projects/skimmer
Using database: /home/orangepi/code_projects/skimmer/data/skimmer.db


In [10]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

if tables_df.empty:
    raise RuntimeError('No tables found in the database.')

display(tables_df)

,table_name
0,bronze_socialblade_channel_profiles
1,bronze_socialblade_channel_stats
2,bronze_vidiq_channel_profiles
3,bronze_vidiq_channel_stats
4,bronze_youtube_skimmed
5,bronze_youtubeapi_channel_stats
6,bronze_youtubeapi_video_stats
7,collection_attempts
8,collection_errors
9,discovery_seed_history


In [11]:
def table_schema(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"PRAGMA table_info({table_name})", connection)


def row_count(table_name: str) -> int:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    return int(pd.read_sql_query(query, connection).iloc[0]['row_count'])


def preview_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    schema = table_schema(table_name)
    column_names = schema['name'].tolist()
    order_column = next(
        (
            column
            for column in ('id', 'last_seen_at', 'observed_at', 'occurred_at', 'created_at', 'latest_video_at')
            if column in column_names
        ),
        next((column for column, is_primary_key in zip(schema['name'], schema['pk']) if is_primary_key), None),
    )
    order_by = f' ORDER BY "{order_column}" DESC' if order_column else ''
    query = f'SELECT * FROM "{table_name}"{order_by} LIMIT {int(limit)}'
    df = pd.read_sql_query(query, connection)

    if 'raw_record_json' in df.columns:
        def parse_raw(value):
            if value is None:
                return None
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value

        df['raw_record_json_parsed'] = df['raw_record_json'].map(parse_raw)

    return df


In [12]:
summary_rows = []
for table_name in tables_df['table_name'].tolist():
    summary_rows.append({'table_name': table_name, 'row_count': row_count(table_name)})

summary_df = pd.DataFrame(summary_rows).sort_values(['row_count', 'table_name'], ascending=[False, True])
display(summary_df)

,table_name,row_count
6,bronze_youtubeapi_video_stats,742145
4,bronze_youtube_skimmed,414856
10,profile_queue,48533
5,bronze_youtubeapi_channel_stats,40256
7,collection_attempts,19436
3,bronze_vidiq_channel_stats,8448
8,collection_errors,8049
2,bronze_vidiq_channel_profiles,7128
9,discovery_seed_history,977
1,bronze_socialblade_channel_stats,311


In [6]:
for table_name in tables_df['table_name'].tolist():
    print('\n' + '=' * 100)
    print(f'TABLE: {table_name}')
    print('=' * 100)

    print('\nSchema:')
    display(table_schema(table_name))

    print('\nLatest rows:')
    display(preview_table(table_name, limit=20))


TABLE: bronze_socialblade_channel_profiles

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,channel_name,TEXT,0,None,0
3,3,channel_type,TEXT,0,None,0
4,4,country,TEXT,0,None,0
5,5,created_at,TEXT,0,None,0
6,6,subscribers_total,,0,None,0
7,7,views_total,,0,None,0
8,8,videos_total,,0,None,0
9,9,data_digest,TEXT,1,None,0



Latest rows:


,id,channel_id,channel_name,channel_type,country,created_at,subscribers_total,views_total,videos_total,data_digest



TABLE: bronze_socialblade_channel_stats

Schema:


...

,channel_key,channel_id,channel_name,latest_video_at,last_seen_at,last_success_at,digested,assigned_source,vidiq_failed,socialblade_failed,needs_review,claimed_by,claimed_at,youtube_channel_id,channel_id_claimed_by,channel_id_claimed_at,youtube_channel_id_attempted_at,youtube_api_failed,youtube_api_attempted_at,youtube_api_success_at
0,@abcnews,@ABCNews,ABC News,2026-08-04T18:07:30+00:00,2026-08-05T02:07:45+00:00,2026-07-30T05:02:24+00:00,1,vidiq,1,1,0,None,None,UCBi2mrWuNuyYy4gbM6fU18Q,None,None,None,0,2026-07-29T02:47:30+00:00,2026-07-29T02:59:58+00:00
1,@allanwv,@allanwv,allanwv,2009-08-09T02:07:45+00:00,2026-08-05T02:07:45+00:00,2026-07-26T03:19:58+00:00,1,vidiq,0,0,0,youtube-api-487577,2026-07-30T16:31:43+00:00,UCcQQ3J0uzK4GO6BKF7KIYKA,None,None,None,1,2026-07-30T16:31:43+00:00,2026-07-26T03:19:58+00:00
2,@allrecipes,@allrecipes,Allrecipes,2026-07-28T17:51:27+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:29:54+00:00,1,vidiq,0,0,0,None,None,UC4tAgeVdaNB5vD_mBoxg50w,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:29:54+00:00
3,@ampexclusive,@AMPEXCLUSIVE,AMP,2026-08-03T16:55:36+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:55:24+00:00,1,vidiq,0,1,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UCJbYdyufHR-cxOuY96KIoqA,None,None,None,1,2026-07-29T02:24:25+00:00,2026-07-27T03:55:24+00:00
4,@andreipiano,@AndreiPiano,Andrei Piano,2023-08-05T10:30:12+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:34:20+00:00,1,vidiq,0,0,0,None,None,UC6oPj0CXwGi1M280hUW-vyw,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:34:20+00:00
5,@asmontv,@AsmonTV,Asmongold TV,2026-08-04T15:56:48+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:51:11+00:00,1,socialblade,1,0,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UCQeRaTukNYft1_6AZPACnog,None,None,2026-07-21T03:57:55+00:00,1,2026-07-29T02:24:25+00:00,2026-07-27T03:51:11+00:00
6,@ballislife,@ballislife,Ballislife,2026-08-03T21:55:00+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:53:25+00:00,1,vidiq,0,0,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UC_zgOsTPdML6tol9hLYh4fQ,None,None,None,1,2026-07-29T02:24:25+00:00,2026-07-27T03:53:25+00:00
7,@baseballbatbros,@baseballbatbros,The Baseball Bat Bros,2026-07-28T16:51:39+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:20:24+00:00,1,socialblade,1,1,0,None,None,UCUC_sJGtV-oKdTw1s3oazwQ,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:20:24+00:00
8,@beardmeatsfood,@Beardmeatsfood,BeardMeatsFood,2026-08-03T17:55:41+00:00,2026-08-05T02:07:45+00:00,2026-07-27T04:31:56+00:00,1,vidiq,1,1,0,youtube-api-296385,2026-07-29T02:47:30+00:00,UCc9CjaAjsMMvaSghZB7-Kog,None,None,None,0,2026-07-29T02:47:30+00:00,2026-07-27T04:31:56+00:00
9,@bingecentral,@BingeCentral,Binge Central,2026-07-22T02:07:45+00:00,2026-08-05T02:07:45+00:00,2026-07-27T04:28:03+00:00,1,socialblade,1,0,0,youtube-api-296385,2026-07-29T02:47:30+00:00,UCSu7x-J8ESHojsq8mCvFOZw,None,None,2026-07-22T06:08:34+00:00,0,2026-07-29T02:47:30+00:00,2026-07-27T04:28:03+00:00



TABLE: youtube_api_quota_usage

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,quota_date,TEXT,0,None,1
1,1,units_used,INTEGER,1,0,0



Latest rows:


,quota_date,units_used
0,2026-08-04,9975
1,2026-08-03,2234
2,2026-08-02,8000
3,2026-08-01,8000
4,2026-07-31,8000
5,2026-07-30,8000
6,2026-07-29,8000
7,2026-07-28,3934
8,2026-07-27,274
9,2026-07-26,8000


In [7]:
display(preview_table('profile_queue'))

,channel_key,channel_id,channel_name,latest_video_at,last_seen_at,last_success_at,digested,assigned_source,vidiq_failed,socialblade_failed,needs_review,claimed_by,claimed_at,youtube_channel_id,channel_id_claimed_by,channel_id_claimed_at,youtube_channel_id_attempted_at,youtube_api_failed,youtube_api_attempted_at,youtube_api_success_at
0,@abcnews,@ABCNews,ABC News,2026-08-04T18:07:30+00:00,2026-08-05T02:07:45+00:00,2026-07-30T05:02:24+00:00,1,vidiq,1,1,0,None,None,UCBi2mrWuNuyYy4gbM6fU18Q,None,None,None,0,2026-07-29T02:47:30+00:00,2026-07-29T02:59:58+00:00
1,@allanwv,@allanwv,allanwv,2009-08-09T02:07:45+00:00,2026-08-05T02:07:45+00:00,2026-07-26T03:19:58+00:00,1,vidiq,0,0,0,youtube-api-487577,2026-07-30T16:31:43+00:00,UCcQQ3J0uzK4GO6BKF7KIYKA,None,None,None,1,2026-07-30T16:31:43+00:00,2026-07-26T03:19:58+00:00
2,@allrecipes,@allrecipes,Allrecipes,2026-07-28T17:51:27+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:29:54+00:00,1,vidiq,0,0,0,None,None,UC4tAgeVdaNB5vD_mBoxg50w,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:29:54+00:00
3,@ampexclusive,@AMPEXCLUSIVE,AMP,2026-08-03T16:55:36+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:55:24+00:00,1,vidiq,0,1,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UCJbYdyufHR-cxOuY96KIoqA,None,None,None,1,2026-07-29T02:24:25+00:00,2026-07-27T03:55:24+00:00
4,@andreipiano,@AndreiPiano,Andrei Piano,2023-08-05T10:30:12+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:34:20+00:00,1,vidiq,0,0,0,None,None,UC6oPj0CXwGi1M280hUW-vyw,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:34:20+00:00
5,@asmontv,@AsmonTV,Asmongold TV,2026-08-04T15:56:48+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:51:11+00:00,1,socialblade,1,0,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UCQeRaTukNYft1_6AZPACnog,None,None,2026-07-21T03:57:55+00:00,1,2026-07-29T02:24:25+00:00,2026-07-27T03:51:11+00:00
6,@ballislife,@ballislife,Ballislife,2026-08-03T21:55:00+00:00,2026-08-05T02:07:45+00:00,2026-07-27T03:53:25+00:00,1,vidiq,0,0,0,youtube-api-292221,2026-07-29T02:24:25+00:00,UC_zgOsTPdML6tol9hLYh4fQ,None,None,None,1,2026-07-29T02:24:25+00:00,2026-07-27T03:53:25+00:00
7,@baseballbatbros,@baseballbatbros,The Baseball Bat Bros,2026-07-28T16:51:39+00:00,2026-08-05T02:07:45+00:00,2026-07-30T17:20:24+00:00,1,socialblade,1,1,0,None,None,UCUC_sJGtV-oKdTw1s3oazwQ,None,None,None,0,2026-07-30T16:31:43+00:00,2026-07-30T17:20:24+00:00
8,@beardmeatsfood,@Beardmeatsfood,BeardMeatsFood,2026-08-03T17:55:41+00:00,2026-08-05T02:07:45+00:00,2026-07-27T04:31:56+00:00,1,vidiq,1,1,0,youtube-api-296385,2026-07-29T02:47:30+00:00,UCc9CjaAjsMMvaSghZB7-Kog,None,None,None,0,2026-07-29T02:47:30+00:00,2026-07-27T04:31:56+00:00
9,@bingecentral,@BingeCentral,Binge Central,2026-07-22T02:07:45+00:00,2026-08-05T02:07:45+00:00,2026-07-27T04:28:03+00:00,1,socialblade,1,0,0,youtube-api-296385,2026-07-29T02:47:30+00:00,UCSu7x-J8ESHojsq8mCvFOZw,None,None,2026-07-22T06:08:34+00:00,0,2026-07-29T02:47:30+00:00,2026-07-27T04:28:03+00:00


In [8]:
# Close last: every cell above queries through this connection.
connection.close()